In [8]:
import mlflow
mlflow.set_tracking_uri("http://ec2-100-53-30-206.compute-1.amazonaws.com:5000/")

In [2]:
# Set or create an experiment
mlflow.set_experiment("Exp 2 - BoW vs TfIdf")

2026/10/03 00:00:11 INFO mlflow.tracking.fluent: Experiment with name 'Exp 2 - BoW vs TfIdf' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlfow-bucket-98/2', creation_time=1790965811433, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1790965811433, lifecycle_stage='active', name='Exp 2 - BoW vs TfIdf', tags={}, trace_location=None, workspace='default'>

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [5]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [7]:
# Step 1: Function to run the experiment
def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):

    # Step 2: Create vectorizer
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(
            ngram_range=ngram_range,
            max_features=vectorizer_max_features
        )
    else:
        vectorizer = TfidfVectorizer(
            ngram_range=ngram_range,
            max_features=vectorizer_max_features
        )

    # Step 3: Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        df["clean_comment"],
        df["category"],
        test_size=0.2,
        random_state=42,
        stratify=df["category"]
    )

    # Step 4: Vectorization
    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 5: Start MLflow run
    with mlflow.start_run() as run:

        # -----------------------------
        # Log run information
        # -----------------------------
        mlflow.set_tag(
            "mlflow.runName",
            f"{vectorizer_name}_{ngram_range}_RandomForest"
        )

        mlflow.set_tag(
            "experiment_type",
            "feature_engineering"
        )

        mlflow.set_tag(
            "model_type",
            "RandomForestClassifier"
        )

        mlflow.set_tag(
            "description",
            f"RandomForest with {vectorizer_name}, "
            f"ngram_range={ngram_range}, "
            f"max_features={vectorizer_max_features}"
        )

        # -----------------------------
        # Log vectorizer parameters
        # -----------------------------
        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", str(ngram_range))
        mlflow.log_param(
            "vectorizer_max_features",
            vectorizer_max_features
        )

        # -----------------------------
        # Random Forest parameters
        # -----------------------------
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("random_state", 42)

        # -----------------------------
        # Train Random Forest
        # -----------------------------
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42
        )

        model.fit(X_train, y_train)

        # -----------------------------
        # Make predictions
        # -----------------------------
        y_pred = model.predict(X_test)

        # -----------------------------
        # Accuracy
        # -----------------------------
        accuracy = accuracy_score(y_test, y_pred)

        mlflow.log_metric("accuracy", accuracy)

        # -----------------------------
        # Classification report
        # -----------------------------
        classification_rep = classification_report(
            y_test,
            y_pred,
            output_dict=True
        )

        for label, metrics in classification_rep.items():

            if isinstance(metrics, dict):

                for metric, value in metrics.items():

                    # Convert label to safe string
                    safe_label = str(label).replace(" ", "_")

                    mlflow.log_metric(
                        f"{safe_label}_{metric}",
                        value
                    )

        # -----------------------------
        # Confusion Matrix
        # -----------------------------
        conf_matrix = confusion_matrix(
            y_test,
            y_pred
        )

        plt.figure(figsize=(8, 6))

        sns.heatmap(
            conf_matrix,
            annot=True,
            fmt="d",
            cmap="Blues"
        )

        plt.xlabel("Predicted")
        plt.ylabel("Actual")

        plt.title(
            f"Confusion Matrix: {vectorizer_name}, {ngram_range}"
        )

        # Save locally
        confusion_matrix_file = (
            f"confusion_matrix_{vectorizer_name}_{ngram_range[0]}_{ngram_range[1]}.png"
        )

        plt.savefig(
            confusion_matrix_file,
            bbox_inches="tight"
        )

        plt.close()

        # Log artifact
        mlflow.log_artifact(
            confusion_matrix_file
        )

        # -----------------------------
        # Log Random Forest model
        # -----------------------------
        mlflow.sklearn.log_model(
            model,
            f"random_forest_model_{vectorizer_name}_{ngram_range}",
            skops_trusted_types=[
                "sklearn.tree._tree.Tree"
            ]
        )

        # -----------------------------
        # Print results
        # -----------------------------
        print(
            f"{vectorizer_name} | "
            f"n-gram={ngram_range} | "
            f"Accuracy={accuracy:.4f}"
        )


# ============================================================
# Step 6: Run experiments
# ============================================================

ngram_ranges = [
    (1, 1),   # Unigram
    (1, 2),   # Bigram
    (1, 3)    # Trigram
]

max_features = 5000


# ============================================================
# Step 7: Run BoW and TF-IDF experiments
# ============================================================

for ngram_range in ngram_ranges:

    # -----------------------------
    # BoW experiment
    # -----------------------------
    run_experiment(
        vectorizer_type="BoW",
        ngram_range=ngram_range,
        vectorizer_max_features=max_features,
        vectorizer_name="BoW"
    )

    # -----------------------------
    # TF-IDF experiment
    # -----------------------------
    run_experiment(
        vectorizer_type="TF-IDF",
        ngram_range=ngram_range,
        vectorizer_max_features=max_features,
        vectorizer_name="TF-IDF"
    )

2026/10/03 00:04:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


BoW | n-gram=(1, 1) | Accuracy=0.6448
🏃 View run BoW_(1, 1)_RandomForest at: http://ec2-100-53-30-206.compute-1.amazonaws.com:5000/#/experiments/2/runs/7ede037253f44247b562fce124a536b4
🧪 View experiment at: http://ec2-100-53-30-206.compute-1.amazonaws.com:5000/#/experiments/2


2026/10/03 00:05:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


TF-IDF | n-gram=(1, 1) | Accuracy=0.6450
🏃 View run TF-IDF_(1, 1)_RandomForest at: http://ec2-100-53-30-206.compute-1.amazonaws.com:5000/#/experiments/2/runs/6d391ef1164f4cb9a9d667352487efc6
🧪 View experiment at: http://ec2-100-53-30-206.compute-1.amazonaws.com:5000/#/experiments/2


2026/10/03 00:06:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

TF-IDF | n-gram=(1, 2) | Accuracy=0.6467
🏃 View run TF-IDF_(1, 2)_RandomForest at: http://ec2-100-53-30-206.compute-1.amazonaws.com:5000/#/experiments/2/runs/95abb1df1b2c4fedbed11ba8a9fe426e
🧪 View experiment at: http://ec2-100-53-30-206.compute-1.amazonaws.com:5000/#/experiments/2


2026/10/03 00:07:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


BoW | n-gram=(1, 3) | Accuracy=0.6547
🏃 View run BoW_(1, 3)_RandomForest at: http://ec2-100-53-30-206.compute-1.amazonaws.com:5000/#/experiments/2/runs/2fe462a30f384c98b868d9c4e2008b67
🧪 View experiment at: http://ec2-100-53-30-206.compute-1.amazonaws.com:5000/#/experiments/2


2026/10/03 00:08:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


TF-IDF | n-gram=(1, 3) | Accuracy=0.6540
🏃 View run TF-IDF_(1, 3)_RandomForest at: http://ec2-100-53-30-206.compute-1.amazonaws.com:5000/#/experiments/2/runs/5d26e9eddd70482eb0a15fb03e3720a8
🧪 View experiment at: http://ec2-100-53-30-206.compute-1.amazonaws.com:5000/#/experiments/2
